In [1]:
import os
import gzip
import bz2
import lzma
import subprocess
import shutil
import numpy as np
import pandas as pd

def compress_file(input_path, method):
    temp_output = input_path + ".compressed"

    if method == "gzip":
        with open(input_path, 'rb') as f_in, gzip.open(temp_output, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    elif method == "bzip2":
        with open(input_path, 'rb') as f_in, bz2.open(temp_output, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    elif method == "lzma":
        with open(input_path, 'rb') as f_in, lzma.open(temp_output, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    elif method == "ppmd":
        temp_output = input_path + ".7z"
        command = ['7z', 'a', '-m0=PPMD', temp_output, input_path]
        subprocess.run(command, check=True)
    else:
        return None

    size_kb = os.path.getsize(temp_output) / 1024
    os.remove(temp_output)
    return size_kb



def compute_entropy_min_size(file_path, bit_depth=8):
    with open(file_path, 'rb') as f:
        data = list(f.read())
    hist = [data.count(i) for i in range(256)]
    ent = -np.sum([p * np.log2(p) for p in np.array(hist) / sum(hist) if p > 0])  # Shannon Entropy
    original_size = os.path.getsize(file_path) / 1024
    min_size = (ent * original_size) / bit_depth
    return min_size

def compute_conditional_entropy(file_path):
    with open(file_path, 'rb') as f:
        data = list(f.read())
    joint_counts = np.zeros((256, 256))
    for i in range(len(data) - 1):
        joint_counts[data[i], data[i + 1]] += 1
    joint_probs = joint_counts / np.sum(joint_counts)
    marginal_probs = np.sum(joint_probs, axis=1, keepdims=True)
    conditional_probs = np.divide(joint_probs, marginal_probs, where=marginal_probs != 0)
    conditional_entropy = -np.nansum(joint_probs * np.log2(conditional_probs, where=conditional_probs > 0))
    original_size = os.path.getsize(file_path) / 1024
    min_size = (conditional_entropy * original_size) / 8
    return min_size

def process_file(file_path):
    original_size = os.path.getsize(file_path) / 1024  # Original file size (KB)
    gzip_size = compress_file(file_path, "gzip")
    bzip2_size = compress_file(file_path, "bzip2")
    lzma_size = compress_file(file_path, "lzma")
    ppmd_size = compress_file(file_path, "ppmd")
    entropy_8 = compute_entropy_min_size(file_path, 8)
    entropy_16 = compute_entropy_min_size(file_path, 16)
    entropy_conditional = compute_conditional_entropy(file_path)

    return {
        "File Name": os.path.basename(file_path),
        "Original Size (KB)": original_size,
        "GZIP Size (KB)": gzip_size,
        "BZIP2 Size (KB)": bzip2_size,
        "LZMA Size (KB)": lzma_size,
        "PPMD Size (KB)": ppmd_size,
        "8-bit Entropy Min Size (KB)": entropy_8,
        "16-bit Entropy Min Size (KB)": entropy_16,
        "8-bit Conditional Entropy Min Size (KB)": entropy_conditional
    }

# 主函数
def main(dataset_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for category in ["Idle", "Physical_Interaction","Power", "Scenario", "Web_Interaction"]:
        category_path = os.path.join(dataset_dir, category)
        if not os.path.exists(category_path):
            print(f"Category path not found: {category_path}")
            continue

        for topology in ["Topology_A", "Topology_B"]:
            topology_path = os.path.join(category_path, topology)
            if not os.path.exists(topology_path):
                print(f"Topology path not found: {topology_path}")
                continue

            topology_results = []

            for root, _, files in os.walk(topology_path):
                for file in files:
                    if file.endswith(".pcapng"):
                        file_path = os.path.join(root, file)
                        result = process_file(file_path)
                        topology_results.append(result)

            if topology_results:
                final_df = pd.DataFrame(topology_results)
                output_file = os.path.join(output_dir, f"{category}_{topology}_results.csv")
                final_df.to_csv(output_file, index=False)
                print(f"Results saved to {output_file}")

dataset_dir = "./Data"
output_dir = "10-Loseless_size_results"
main(dataset_dir, output_dir)


Results saved to ./Data/Loseless_size_results\Idle_Topology_A_results.csv
Results saved to ./Data/Loseless_size_results\Idle_Topology_B_results.csv
Results saved to ./Data/Loseless_size_results\Physical_Interaction_Topology_A_results.csv
Results saved to ./Data/Loseless_size_results\Physical_Interaction_Topology_B_results.csv
Results saved to ./Data/Loseless_size_results\Power_Topology_A_results.csv


C:\Users\郑大林\AppData\Local\Temp\ipykernel_8500\581750841.py:53: RuntimeWarning: invalid value encountered in multiply
  conditional_entropy = -np.nansum(joint_probs * np.log2(conditional_probs, where=conditional_probs > 0))


Results saved to ./Data/Loseless_size_results\Power_Topology_B_results.csv
Results saved to ./Data/Loseless_size_results\Scenario_Topology_A_results.csv
Results saved to ./Data/Loseless_size_results\Scenario_Topology_B_results.csv
Results saved to ./Data/Loseless_size_results\Web_Interaction_Topology_A_results.csv
Results saved to ./Data/Loseless_size_results\Web_Interaction_Topology_B_results.csv


In [8]:
import os
import pandas as pd
import numpy as np

def process_files(folder_path):
    merged_a = None
    merged_b = None

    for file in os.listdir(folder_path):
        if file.endswith('.csv'):
            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path)

            for col in df.columns:
                if col != 'File Name':
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            if '_Topology_A_' in file:
                merged_a = df if merged_a is None else pd.concat([merged_a, df], axis=0, ignore_index=True)
            elif '_Topology_B_' in file:
                merged_b = df if merged_b is None else pd.concat([merged_b, df], axis=0, ignore_index=True)

    if merged_a is not None:
        process_single_topology(merged_a, 'Topology_A')

    if merged_b is not None:
        process_single_topology(merged_b, 'Topology_B')


def process_single_topology(df, topology_name):
    if 'Original Size (KB)' not in df.columns:
        return

    original_size = df['Original Size (KB)']
    result_df = df.copy()

    for col in df.columns:
        if col not in ['Original Size (KB)', 'File Name']:
            result_df[f'{col}_ratio'] = original_size/df[col]
    ratio_columns = [col for col in result_df.columns if col.endswith('_ratio')]

    stats = pd.DataFrame({
        'Column': ratio_columns,
        'Mean': [result_df[col].mean() for col in ratio_columns],
        'Variance': [result_df[col].var() for col in ratio_columns]
    })

    mean_row = {col: stats.loc[stats['Column'] == col, 'Mean'].values[0] if col in ratio_columns else
                'Mean' if col == 'file name' else np.nan
                for col in result_df.columns}

    var_row = {col: stats.loc[stats['Column'] == col, 'Variance'].values[0] if col in ratio_columns else
               'Variance' if col == 'file name' else np.nan
               for col in result_df.columns}

    result_df = pd.concat([
        result_df,
        pd.DataFrame([mean_row]),
        pd.DataFrame([var_row])
    ], ignore_index=True)

    output_file = f'merged_{topology_name}_with_ratios.csv'
    result_df.to_csv(output_file, index=False)
    print(f"Saved {topology_name} to {output_file}")

folder_path = "10-Loseless_size_results"
process_files(folder_path)


Saved Topology_A to merged_Topology_A_with_ratios.csv
Saved Topology_B to merged_Topology_B_with_ratios.csv
